In [ ]:
# NOTEBOOK NAME
# FeatureStatsSandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd

# # from pathlib import Path      # used to play with pathnames to save

# # from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER


# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *
# sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')  # folder *containing* pyflextrkr/

# import logging
# logging.basicConfig(level=logging.INFO)
# from pyflextrkr.idcells_reflectivity import idcells_reflectivity
# from pyflextrkr.tracksingle_driver import tracksingle_driver
# from pyflextrkr.gettracks import gettracknumbers
# from pyflextrkr.trackstats_driver import trackstats_driver

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

# import glob

# import shutil # I think this is to rename the output file from PyFLEXTRKR
# import gc # something to prevent memory leaks

# from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

# import matplotlib.colors as mcolors
# import matplotlib.cm as cm

# # from matplotlib.patches import Circle # for radar range ring circles on the map
from matplotlib.lines import Line2D # for plotting stars on the map legend

In [ ]:
# FEATURE DATA LOADING


# get rid of them BS warnings I don't care about
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='argopy')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='xarray')


RadarIDno              = 22           # [see Radar Number Catalogue below (int)]                   # Which radar site are you plotting data for?
PlotDate               = '2024-03-09' # ['YYYY-MM-DD' (string)]                                    # Which (UTC) date are you plotting for?

# retrieve pieces of the date and construct one without dashes
YearStr = PlotDate[0:4]
MonthStr = PlotDate[5:7]
DayStr = PlotDate[8:10]
FileDateStr = YearStr + MonthStr + DayStr

FeatureStoragePath = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/V35/{FileDateStr}/{RadarIDno}_{FileDateStr}FeatureStats.nc'
FeatureXR = xr.open_dataset(FeatureStoragePath)

# Create a new time dimension for time of day, instead of just time in a given feature's life
# Generate 288 x 5-minute timestamps to assign as coordinate along the 'times' dimension
RadarFileDatePD = pd.Timestamp(PlotDate).date()
FrameTimes = pd.date_range(start=pd.Timestamp(RadarFileDatePD), periods=288, freq='5min')
FeatureXR = FeatureXR.assign_coords(FrameTimes=xr.DataArray(FrameTimes, dims='FrameTimes'))

# add variables that rearrange times by time of day rather than time in a given feature's life
FeatureXR = AddFrameTimeVars(FeatureXR)



In [ ]:
# FEATURE DATA LOADING VERSION 2


# get rid of them BS warnings I don't care about
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='argopy')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='xarray')


# RadarIDno              = 41           # [see Radar Number Catalogue below (int)]                   # Which radar site are you plotting data for?
PlotDate               = '2024-03-09' # ['YYYY-MM-DD' (string)]                                    # Which (UTC) date are you plotting for?

# retrieve pieces of the date and construct one without dashes
YearStr = PlotDate[0:4]
MonthStr = PlotDate[5:7]
DayStr = PlotDate[8:10]
FileDateStr = YearStr + MonthStr + DayStr

FeatureStoragePath1 = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/22/V5/{FileDateStr}/22_{FileDateStr}FeatureStats.nc'
FeatureXR1 = xr.open_dataset(FeatureStoragePath1)

FeatureStoragePath2 = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/41/V5/{FileDateStr}/41_{FileDateStr}FeatureStats.nc'
FeatureXR2 = xr.open_dataset(FeatureStoragePath2)

# Create a new time dimension for time of day, instead of just time in a given feature's life
# Generate 288 x 5-minute timestamps to assign as coordinate along the 'times' dimension
RadarFileDatePD = pd.Timestamp(PlotDate).date()
FrameTimes = pd.date_range(start=pd.Timestamp(RadarFileDatePD), periods=288, freq='5min')
FeatureXR1 = FeatureXR1.assign_coords(FrameTimes=xr.DataArray(FrameTimes, dims='FrameTimes'))
FeatureXR2 = FeatureXR2.assign_coords(FrameTimes=xr.DataArray(FrameTimes, dims='FrameTimes'))

# add variables that rearrange times by time of day rather than time in a given feature's life
FeatureXR1 = AddFrameTimeVars(FeatureXR1)
FeatureXR2 = AddFrameTimeVars(FeatureXR2)

In [ ]:
FeatureXR

In [ ]:
# CHAD PLOT
# FEATURE LIFETIME TRAJECTORIES

# --- User inputs ---
variable        = 'core_area'   # <-- change to your chosen variable string
MinimumFrameCount = 3                      # <-- minimum valid frames for a track to be plotted

# --- Extract data ---
data         = FeatureXR[variable]           # shape: (tracks, times)
times        = FeatureXR.coords['times'].values
time_minutes = times * 5                   # convert frames to minutes

# string for variable units
VariableUnits = FeatureXR[variable].attrs['units']

# --- Determine which tracks pass the minimum frame count ---
valid_tracks      = []
valid_track_times = []   # list of arrays: valid time_minutes for each passing track
valid_track_vals  = []   # list of arrays: valid variable values for each passing track

for track_idx in range(len(FeatureXR.coords['tracks'])):
    track_data  = data.isel(tracks=track_idx).values
    valid_mask  = ~np.isnan(track_data)
    valid_count = np.sum(valid_mask)

    if valid_count < MinimumFrameCount:
        continue

    valid_tracks.append(track_idx)
    valid_track_times.append(time_minutes[valid_mask])
    valid_track_vals.append(track_data[valid_mask])

# --- Compute x-axis range ---
max_time_plotted = max(vt[-1] for vt in valid_track_times) if valid_tracks else 0
x_buffer         = 2.5   # half a frame width in minutes
x_min            = 0 - x_buffer
x_max            = max_time_plotted + x_buffer

# --- Compute histogram: number of features alive at each time frame ---
# A feature is counted at a given time frame if it has a valid value there
unique_times    = time_minutes                          # all possible time frames
feature_counts  = np.zeros(len(unique_times), dtype=int)

for vt in valid_track_times:
    for t in vt:
        idx = np.where(unique_times == t)[0]
        if len(idx) > 0:
            feature_counts[idx[0]] += 1

# Only keep time frames within the plotted x range (excluding buffer)
hist_mask    = (unique_times >= 0) & (unique_times <= max_time_plotted)
hist_times   = unique_times[hist_mask]
hist_counts  = feature_counts[hist_mask]

hist_max = int(np.ceil(np.max(hist_counts) / 100) * 100) if np.any(hist_counts > 0) else 100

# --- Set up figure with gridspec: line plot on top, histogram on bottom ---
fig = plt.figure(figsize=(20, 12))
gs  = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.05)

ax_lines = fig.add_subplot(gs[0])
ax_hist  = fig.add_subplot(gs[1], sharex=ax_lines)

fig.patch.set_facecolor('black')
ax_lines.set_facecolor('black')
ax_hist.set_facecolor('black')

# --- Plot one line per passing track ---
for vt, vv in zip(valid_track_times, valid_track_vals):
    ax_lines.plot(
        vt, vv,
        color=[1,0,0],
        linewidth=0.8,
        alpha=1.0
    )

# --- Line plot axes formatting ---
ax_lines.set_xlim(x_min, x_max)
ax_lines.set_ylabel(variable + ' [' + VariableUnits + ']', color='white', fontsize=13)
ax_lines.tick_params(colors='white', which='both', labelsize=11)
ax_lines.grid(which='major', color='white', linewidth=0.8, linestyle='-', alpha=0.6)
ax_lines.grid(which='minor', color='white', linewidth=0.3, linestyle='-', alpha=0.3)
ax_lines.minorticks_on()
plt.setp(ax_lines.get_xticklabels(), visible=False)

for spine in ax_lines.spines.values():
    spine.set_edgecolor('white')

# --- Bottom histogram ---
ax_hist.bar(
    hist_times,
    hist_counts,
    width=0.8 * 5,          # 0.8 of a 5-minute frame width, matching example style
    color=[0.0, 0.7, 0.7],
    edgecolor='white',
    alpha=1.0  
)

ax_hist.set_xlim(x_min, x_max)
ax_hist.set_ylim(0, hist_max)
ax_hist.set_xlabel('Time (minutes)', color='white', fontsize=13)
ax_hist.set_ylabel('Feature\nCount',  color='white', fontsize=11)
ax_hist.tick_params(colors='white', which='both', labelsize=11)
ax_hist.set_facecolor('black')

for t in hist_times:
    ax_hist.axvline(t, color='white', linewidth=0.2, alpha=0.6)

for spine in ax_hist.spines.values():
    spine.set_edgecolor('white')

# Grid lines on histogram: thin every 100, thick every 500
for v in np.arange(0, hist_max + 1, 100):
    ax_hist.axhline(v, color='white', linewidth=0.2, alpha=0.6)
for v in np.arange(0, hist_max + 1, 500):
    ax_hist.axhline(v, color='white', linewidth=0.5, alpha=0.8)

# --- Align axes widths after rendering ---
plt.tight_layout()
plt.draw()

ax_lines_pos = ax_lines.get_position()
ax_hist.set_position([
    ax_lines_pos.x0,
    ax_hist.get_position().y0,
    ax_lines_pos.width,
    ax_hist.get_position().height
])

plt.show()

In [ ]:
# CHAD PLOT
# FEATURE LIFETIME TRAJECTORIES - PERCENTILE VERSION

# --- User inputs ---
variable          = 'max_dbz'   # <-- change to your chosen variable string
MinimumFrameCount = 120/5             # <-- minimum valid frames for a track to be plotted
LineColour        = [1, 0, 0]     # <-- RGB colour for lines and fills
MinSampleSize     = 10            # <-- minimum number of tracks at a time step to plot stats

# --- Extract data ---
data         = FeatureXR[variable]
times        = FeatureXR.coords['times'].values
time_minutes = times * 5

# string for variable units
VariableUnits = FeatureXR[variable].attrs['units']

# --- Determine which tracks pass the minimum frame count ---
valid_tracks      = []
valid_track_times = []
valid_track_vals  = []

for track_idx in range(len(FeatureXR.coords['tracks'])):
    track_data  = data.isel(tracks=track_idx).values
    valid_mask  = ~np.isnan(track_data)
    valid_count = np.sum(valid_mask)

    if valid_count < MinimumFrameCount:
        continue

    valid_tracks.append(track_idx)
    valid_track_times.append(time_minutes[valid_mask])
    valid_track_vals.append(track_data[valid_mask])

# --- Compute x-axis range ---
max_time_plotted = max(vt[-1] for vt in valid_track_times) if valid_tracks else 0
x_buffer         = 2.5
x_min            = 0 - x_buffer
x_max            = max_time_plotted + x_buffer

# --- Compute percentiles at each time frame ---
unique_times = time_minutes
pct_times    = []
pct_10       = []
pct_25       = []
pct_50       = []
pct_75       = []
pct_90       = []

for t in unique_times:
    # Gather all valid values across tracks at this time frame
    vals_at_t = []
    for vt, vv in zip(valid_track_times, valid_track_vals):
        match = np.where(vt == t)[0]
        if len(match) > 0:
            vals_at_t.append(vv[match[0]])

    if len(vals_at_t) < MinSampleSize:
        continue

    vals_at_t = np.array(vals_at_t)
    pct_times.append(t)
    pct_10.append(np.percentile(vals_at_t, 10))
    pct_25.append(np.percentile(vals_at_t, 25))
    pct_50.append(np.percentile(vals_at_t, 50))
    pct_75.append(np.percentile(vals_at_t, 75))
    pct_90.append(np.percentile(vals_at_t, 90))

pct_times = np.array(pct_times)
pct_10    = np.array(pct_10)
pct_25    = np.array(pct_25)
pct_50    = np.array(pct_50)
pct_75    = np.array(pct_75)
pct_90    = np.array(pct_90)

# --- Compute histogram: number of features alive at each time frame ---
feature_counts = np.zeros(len(unique_times), dtype=int)

for vt in valid_track_times:
    for t in vt:
        idx = np.where(unique_times == t)[0]
        if len(idx) > 0:
            feature_counts[idx[0]] += 1

hist_mask   = (unique_times >= 0) & (unique_times <= max_time_plotted)
hist_times  = unique_times[hist_mask]
hist_counts = feature_counts[hist_mask]

hist_max = int(np.ceil(np.max(hist_counts) / 100) * 100) if np.any(hist_counts > 0) else 100

# --- Set up figure ---
fig = plt.figure(figsize=(20, 12))
gs  = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.05)

ax_lines = fig.add_subplot(gs[0])
ax_hist  = fig.add_subplot(gs[1], sharex=ax_lines)

fig.patch.set_facecolor('black')
ax_lines.set_facecolor('black')
ax_hist.set_facecolor('black')

# --- Plot filled regions ---
# 10-90th percentile: light fill
ax_lines.fill_between(
    pct_times, pct_10, pct_90,
    color=LineColour,
    alpha=0.15,
    linewidth=0
)

# 25-75th percentile: medium fill
ax_lines.fill_between(
    pct_times, pct_25, pct_75,
    color=LineColour,
    alpha=0.35,
    linewidth=0
)

# --- Plot percentile lines ---
ax_lines.plot(pct_times, pct_10, color=LineColour, linewidth=0.8, alpha=0.8, linestyle='--')
ax_lines.plot(pct_times, pct_90, color=LineColour, linewidth=0.8, alpha=0.8, linestyle='--')
ax_lines.plot(pct_times, pct_25, color=LineColour, linewidth=1.5, alpha=0.9, linestyle='-')
ax_lines.plot(pct_times, pct_75, color=LineColour, linewidth=1.5, alpha=0.9, linestyle='-')
ax_lines.plot(pct_times, pct_50, color=LineColour, linewidth=2.5, alpha=1.0, linestyle='-')

# --- Line plot axes formatting ---
ax_lines.set_xlim(x_min, x_max)
ax_lines.set_ylabel(variable + ' [' + VariableUnits + ']', color='white', fontsize=13)
ax_lines.set_title(variable + ' Quantiles Throughout Feature Lifetime for Those Lasting at Least ' + str(int((MinimumFrameCount-1)*5)) + ' Minutes', color='white', fontsize=14)
ax_lines.tick_params(colors='white', which='both', labelsize=11)
ax_lines.grid(which='major', color='white', linewidth=0.8, linestyle='-', alpha=0.6)
ax_lines.grid(which='minor', color='white', linewidth=0.3, linestyle='-', alpha=0.3)
ax_lines.minorticks_on()
plt.setp(ax_lines.get_xticklabels(), visible=False)

for spine in ax_lines.spines.values():
    spine.set_edgecolor('white')

# --- Bottom histogram ---
ax_hist.bar(
    hist_times,
    hist_counts,
    width=0.8 * 5,
    color=[0.0, 0.7, 0.7],
    edgecolor='white',
    alpha=1.0
)

ax_hist.set_xlim(x_min, x_max)
ax_hist.set_ylim(0, hist_max)
ax_hist.set_xlabel("Time Since 'Feature Birth' (minutes)", color='white', fontsize=13)
ax_hist.set_ylabel('Feature\nCount',  color='white', fontsize=11)
ax_hist.tick_params(colors='white', which='both', labelsize=11)
ax_hist.set_facecolor('black')

for t in hist_times:
    ax_hist.axvline(t, color='white', linewidth=0.2, alpha=0.6)

for spine in ax_hist.spines.values():
    spine.set_edgecolor('white')

for v in np.arange(0, hist_max + 1, 100):
    ax_hist.axhline(v, color='white', linewidth=0.2, alpha=0.6)
for v in np.arange(0, hist_max + 1, 500):
    ax_hist.axhline(v, color='white', linewidth=0.5, alpha=0.8)

# --- Align axes widths after rendering ---
plt.tight_layout()
plt.draw()

ax_lines_pos = ax_lines.get_position()
ax_hist.set_position([
    ax_lines_pos.x0,
    ax_hist.get_position().y0,
    ax_lines_pos.width,
    ax_hist.get_position().height
])

plt.show()


In [ ]:
FeatureXR2

In [ ]:
# CHAD PLOT
# FEATURE LIFETIME TRAJECTORIES - PERCENTILE VERSION - DUAL DATASET

# --- User inputs ---
variable          = 'maxETH_30dbz'
MinimumMinutesCount = 15
MinimumFrameCount = (MinimumMinutesCount / 5) + 1  # needs to be in 4 frames to last 15 min etc.
LineColour1       = [1, 0, 0]   # red for dataset 1
LineColour2       = [0, 0, 1]   # blue for dataset 2
MinSampleSize     = 10

Label1 = 'Mackay 2024-03-09'   # <-- change to your chosen label
Label2 = 'Willis Island 2024-03-09'   # <-- change to your chosen label


# --- Helper function to process a dataset ---
def process_dataset(FeatureXR, variable, MinimumFrameCount, MinSampleSize):

    data         = FeatureXR[variable]
    times        = FeatureXR.coords['times'].values
    time_minutes = times * 5

    valid_tracks      = []
    valid_track_times = []
    valid_track_vals  = []

    for track_idx in range(len(FeatureXR.coords['tracks'])):
        track_data  = data.isel(tracks=track_idx).values
        valid_mask  = ~np.isnan(track_data)
        valid_count = np.sum(valid_mask)

        if valid_count < MinimumFrameCount:
            continue

        valid_tracks.append(track_idx)
        valid_track_times.append(time_minutes[valid_mask])
        valid_track_vals.append(track_data[valid_mask])

    # --- Percentiles ---
    unique_times = time_minutes
    pct_times    = []
    pct_10       = []
    pct_25       = []
    pct_50       = []
    pct_75       = []
    pct_90       = []

    for t in unique_times:
        vals_at_t = []
        for vt, vv in zip(valid_track_times, valid_track_vals):
            match = np.where(vt == t)[0]
            if len(match) > 0:
                vals_at_t.append(vv[match[0]])

        if len(vals_at_t) < MinSampleSize:
            continue

        vals_at_t = np.array(vals_at_t)
        pct_times.append(t)
        pct_10.append(np.percentile(vals_at_t, 10))
        pct_25.append(np.percentile(vals_at_t, 25))
        pct_50.append(np.percentile(vals_at_t, 50))
        pct_75.append(np.percentile(vals_at_t, 75))
        pct_90.append(np.percentile(vals_at_t, 90))

    # --- Histogram counts ---
    feature_counts = np.zeros(len(unique_times), dtype=int)
    for vt in valid_track_times:
        for t in vt:
            idx = np.where(unique_times == t)[0]
            if len(idx) > 0:
                feature_counts[idx[0]] += 1

    max_time = max(vt[-1] for vt in valid_track_times) if valid_tracks else 0
    hist_mask   = (unique_times >= 0) & (unique_times <= max_time)
    hist_times  = unique_times[hist_mask]
    hist_counts = feature_counts[hist_mask]

    return (
        np.array(pct_times),
        np.array(pct_10), np.array(pct_25), np.array(pct_50),
        np.array(pct_75), np.array(pct_90),
        hist_times, hist_counts, max_time
    )

# --- Process both datasets ---
(pct_times1, pct_10_1, pct_25_1, pct_50_1, pct_75_1, pct_90_1,
 hist_times1, hist_counts1, max_time1) = process_dataset(
    FeatureXR1, variable, MinimumFrameCount, MinSampleSize)

(pct_times2, pct_10_2, pct_25_2, pct_50_2, pct_75_2, pct_90_2,
 hist_times2, hist_counts2, max_time2) = process_dataset(
    FeatureXR2, variable, MinimumFrameCount, MinSampleSize)

# --- Units (from dataset 1, same variable so same units) ---
VariableUnits = FeatureXR1[variable].attrs['units']

# --- Compute shared x-axis range ---
max_time_plotted = max(max_time1, max_time2)
x_buffer         = 2.5
x_min            = 0 - x_buffer
x_max            = max_time_plotted + x_buffer

# --- Compute shared histogram y-axis max ---
all_counts = np.concatenate([hist_counts1, hist_counts2])
hist_max   = int(np.ceil(np.max(all_counts) / 100) * 100) if len(all_counts) > 0 else 100

# --- Set up figure ---
fig = plt.figure(figsize=(10, 6))
gs  = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.05)

ax_lines = fig.add_subplot(gs[0])
ax_hist  = fig.add_subplot(gs[1], sharex=ax_lines)

fig.patch.set_facecolor('black')
ax_lines.set_facecolor('black')
ax_hist.set_facecolor('black')

# --- Plotting helper for one dataset's percentile lines and fills ---
def plot_percentiles(ax, pct_times, pct_10, pct_25, pct_50, pct_75, pct_90, colour):
    ax.fill_between(pct_times, pct_10, pct_90,
                    color=colour, alpha=0.14, linewidth=0)
    ax.fill_between(pct_times, pct_25, pct_75,
                    color=colour, alpha=0.20, linewidth=0)
    ax.plot(pct_times, pct_10, color=colour, linewidth=0.5, alpha=0.6, linestyle='--')
    ax.plot(pct_times, pct_90, color=colour, linewidth=0.5, alpha=0.6, linestyle='--')
    ax.plot(pct_times, pct_25, color=colour, linewidth=1.0, alpha=0.6, linestyle='-')
    ax.plot(pct_times, pct_75, color=colour, linewidth=1.0, alpha=0.6, linestyle='-')
    ax.plot(pct_times, pct_50, color=colour, linewidth=2.5, alpha=1.0, linestyle='-')

plot_percentiles(ax_lines, pct_times1, pct_10_1, pct_25_1, pct_50_1, pct_75_1, pct_90_1, LineColour1)
plot_percentiles(ax_lines, pct_times2, pct_10_2, pct_25_2, pct_50_2, pct_75_2, pct_90_2, LineColour2)

# --- Line plot axes formatting ---
ax_lines.set_xlim(x_min, x_max)
ax_lines.set_ylabel(variable + ' [' + VariableUnits + ']', color='white', fontsize=13)
ax_lines.set_title(
    variable + ' Quantiles Throughout Feature Lifetime for Those Lasting at Least ' +
    str(int(MinimumMinutesCount)) + ' Minutes',
    color='white', fontsize=14
)
ax_lines.tick_params(colors='white', which='both', labelsize=11)
ax_lines.grid(which='major', color='white', linewidth=0.8, linestyle='-', alpha=0.6)
ax_lines.grid(which='minor', color='white', linewidth=0.3, linestyle='-', alpha=0.3)
ax_lines.minorticks_on()
plt.setp(ax_lines.get_xticklabels(), visible=False)

for spine in ax_lines.spines.values():
    spine.set_edgecolor('white')

legend_elements = [
    Line2D([0], [0], color=LineColour1, linewidth=2.5, label=Label1),
    Line2D([0], [0], color=LineColour2, linewidth=2.5, label=Label2)
]

ax_lines.legend(
    handles=legend_elements,
    facecolor='black',
    edgecolor='white',
    labelcolor='white',
    fontsize=12
)

# --- Bottom histogram: both datasets overlapping with alpha=0.5 ---
ax_hist.bar(hist_times1, hist_counts1,
            width=0.8 * 5, color=LineColour1, edgecolor='white', alpha=0.5)
ax_hist.bar(hist_times2, hist_counts2,
            width=0.8 * 5, color=LineColour2, edgecolor='white', alpha=0.5)

ax_hist.set_xlim(x_min, x_max)
ax_hist.set_ylim(0, hist_max)
ax_hist.set_xlabel("Time Since 'Feature Birth' (minutes)", color='white', fontsize=13)
ax_hist.set_ylabel('Feature\nCount', color='white', fontsize=11)
ax_hist.tick_params(colors='white', which='both', labelsize=11)
ax_hist.set_facecolor('black')

for t in np.union1d(hist_times1, hist_times2):
    ax_hist.axvline(t, color='white', linewidth=0.2, alpha=0.6)

for spine in ax_hist.spines.values():
    spine.set_edgecolor('white')

for v in np.arange(0, hist_max + 1, 100):
    ax_hist.axhline(v, color='white', linewidth=0.2, alpha=0.6)
for v in np.arange(0, hist_max + 1, 500):
    ax_hist.axhline(v, color='white', linewidth=0.5, alpha=0.8)

# --- Align axes widths after rendering ---
plt.tight_layout()
plt.draw()

ax_lines_pos = ax_lines.get_position()
ax_hist.set_position([
    ax_lines_pos.x0,
    ax_hist.get_position().y0,
    ax_lines_pos.width,
    ax_hist.get_position().height
])

# plt.show()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/FeatureLifetimes/'

SaveFile   = FileDateStr + '_' + variable + '_FeatureLifeTimePercentiles_' + str(MinimumMinutesCount) + 'MinMin.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='k', dpi = 300)


In [ ]:
# CHAD PLOT
# SCATTER PLOT BETWEEN TWO CHOSEN VARIABLES WITH COLOUR AS ANOTHER VARIABLE

# ---------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------
XVarName     = 'core_area_frametimes'       # string name of the x-axis variable
YVarName     = 'max_dbz_frametimes'     # string name of the y-axis variable
ColourVarName = 'cell_area_frametimes'  # string name of the colour variable, or None for no colouring

ColourMap    = make_ChadMapZ()       # colourmap to use if colouring by a third variable
PlotColourStyle = 'Dark'             # 'Light' or 'Dark'
# ---------------------------------------------------------------

StandOutColour = 'white'
VarName =            '30 dBZEcho Top Height'
VarNameLong =        'corrected_reflectivity'
VarMinVal =           148      # [dBZ]
VarMaxVal =           150       # [dBZ]
VarUnit =            'km'
VarFillValue =       -32.0     # [dBZ]
VarColourBar =       make_ChadMapZ()
VarColourBar_min =   147
VarColourBar_max =   153
VarColourBar_norm =  Normalize(vmin=147, vmax=153)
VarTickSpacing =     0.5



# Pull the two main variables and flatten to 1D
XData = FeatureXR[XVarName].values.flatten()
YData = FeatureXR[YVarName].values.flatten()

# Build a valid-data mask — only keep points where BOTH x and y are finite
ValidMask = np.isfinite(XData) & np.isfinite(YData)

# If a colour variable is given, also require it to be finite
if ColourVarName is not None:
    CData     = FeatureXR[ColourVarName].values.flatten()
    ValidMask = ValidMask & np.isfinite(CData)
    CDataValid = CData[ValidMask]
else:
    CDataValid = None

# Apply the mask
XDataValid = XData[ValidMask]
YDataValid = YData[ValidMask]

print(f'{np.sum(ValidMask)} valid points out of {len(XData)} total')

# ---------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

if PlotColourStyle == 'Dark':
    fig.patch.set_facecolor('#0a0a0a')
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
else:
    StandOutColour = 'black'

# Scatter plot
if CDataValid is not None:
    sc = ax.scatter(XDataValid, YDataValid, c=CDataValid, cmap=ColourMap, norm=VarColourBar_norm,
                    s=5, alpha=0.8, linewidths=0)

     # GridViewer = ax.pcolormesh( PlotLons, PlotLats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = fig.colorbar(sc, ax=ax)
    # colour bar controls
    # cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
    cbar.ax.yaxis.set_tick_params(color = StandOutColour)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing
else:
    ax.scatter(XDataValid, YDataValid, s=5, alpha=0.8, linewidths=0,
               color=StandOutColour)

# Labels and styling
ax.set_xlabel(XVarName, color=StandOutColour)
ax.set_ylabel(YVarName, color=StandOutColour)
ax.tick_params(colors=StandOutColour)
for spine in ax.spines.values():
    spine.set_edgecolor(StandOutColour)

ax.set_title(f'{YVarName} vs {XVarName}', color=StandOutColour)

plt.grid()
# plt.xlim([0,500])
# plt.ylim([0,60])

plt.tight_layout()
plt.show()

In [ ]:
# CHAD PLOT
# SCATTER PLOT BETWEEN TWO CHOSEN VARIABLES AS A 2D DENSITY PLOT

# ---------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------
XVarName      = 'core_area'    # string name of the x-axis variable
YVarName      = 'max_dbz'      # string name of the y-axis variable
DensityColourMap = plt.cm.plasma          # colourmap for the density plot
XBins         = 100                       # number of bins in x
YBins         = 100                       # number of bins in y
PlotColourStyle = 'Dark'                  # 'Light' or 'Dark'
# ---------------------------------------------------------------

# Pull the two variables and flatten to 1D
XData = FeatureXR[XVarName].values.flatten()
YData = FeatureXR[YVarName].values.flatten()

# Build a valid-data mask
ValidMask  = np.isfinite(XData) & np.isfinite(YData)
XDataValid = XData[ValidMask]
YDataValid = YData[ValidMask]
TotalValid = np.sum(ValidMask)

print(f'{TotalValid} valid points out of {len(XData)} total')

# --- Compute 2D histogram ---
x_edges = np.linspace(XDataValid.min(), XDataValid.max(), XBins + 1)
y_edges = np.linspace(YDataValid.min(), YDataValid.max(), YBins + 1)

h, x_edges, y_edges = np.histogram2d(XDataValid, YDataValid, bins=[x_edges, y_edges])

# Convert to percentage of total valid points
h = (h / TotalValid) * 100

# Mask empty cells
h[h == 0] = np.nan

# --- Plotting ---
fig, ax = plt.subplots(figsize=(8, 6))

if PlotColourStyle == 'Dark':
    fig.patch.set_facecolor('#0a0a0a')
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
else:
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    StandOutColour = 'black'

# pcolormesh density plot
mesh = ax.pcolormesh(x_edges, y_edges, h.T, cmap=DensityColourMap, shading='auto')

# Colourbar
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('% of valid points', color=StandOutColour)
cbar.ax.yaxis.set_tick_params(color=StandOutColour)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=StandOutColour)

# Labels and styling
ax.set_xlabel(XVarName, color=StandOutColour, fontsize=12)
ax.set_ylabel(YVarName, color=StandOutColour, fontsize=12)
ax.set_title(f'{YVarName} vs {XVarName}', color=StandOutColour, fontsize=13)
ax.tick_params(colors=StandOutColour)

for spine in ax.spines.values():
    spine.set_edgecolor(StandOutColour)

ax.grid(which='major', color=StandOutColour, linewidth=0.4, linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# --- Load PyFLEXTRKR cell identification file for this timestep ---

RadarIDno = 41
RadarFileTime = '120000'
RadarFileTimePrint = '12:00'
FileDateStr = '20240309'

CellIDPath = (
    '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/SingleTimes/'
    + str(RadarIDno) + '/V5/' + FileDateStr + '/'
    + f'cellidfile_{FileDateStr}_{RadarFileTime}.nc'
)
if os.path.exists(CellIDPath):
    ds_cellid = xr.open_dataset(CellIDPath)
    HasCellID = True
else:
    print(f'No cell ID file found for {RadarFileTimePrint}, skipping Steiner overlay')
    HasCellID = False

In [ ]:
ds_cellid

In [ ]:
 # create the figure to be plotted on with a map projection
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# set the plot background to black and text to white if the user wants it
if (PlotColourStyle=='Dark'):
    fig.patch.set_facecolor('#0a0a0a')  
    ax.set_facecolor('#0a0a0a')
    StandOutColour = 'white'
elif (PlotColourStyle=='Light'):
    StandOutColour = 'black'
else:
    print("PlotColourStyle must be exactly 'Light' or 'Dark'")

# add a background topo map to the plot
ElevationModelPath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
# Extract plot bounds from radar grid
# LonMin, LonMax = float(RadarXR.lon.min()) + LonShift, float(RadarXR.lon.max()) + LonShift
# LatMin, LatMax = float(RadarXR.lat.min()), float(RadarXR.lat.max())

LonMin, LonMax = 145, 150
LatMin, LatMax = -24, -19

# call the "adding-the-map" function
AddTopoShading(ax, LonMin, LonMax, LatMin, LatMax, ElevationModelPath, PlotColourStyle)


# PLOT THE MAIN VARIABLE
# if (DataSourceType == 'Grid'):
    # # index in the netcdf altitude variable for the altitude you want
    # alti = np.where(RadarXR.z == Altitude) # this is a double nested array for some reason
    # alti = alti[0][0] # take the index out of the double nested array

    # # quit out if the altitude does not correspond to one in the netCDF file
    # if ( np.size(alti) != 1): 
    #     raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')
    
    # PlottingArray = RadarXR[VarNameLong][0,alti,:,:]
    # GridViewer = ax.pcolormesh(RadarXR.lon+ LonShift, RadarXR.lat, PlottingArray, 
    #                            cmap=VarColourBar , norm=VarColourBar_norm, transform=ccrs.PlateCarree())

    # # plot a star for the location of the radar on the map
    # ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
    #         marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
    # ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
    #         marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)

    # plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
    #     PlotDate + ' at ' + \
    #     str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC', color = StandOutColour)

# # colour bar controls
# cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
# cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
# cbar.ax.yaxis.set_tick_params(color = StandOutColour)
# plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
# cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
# cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing

# ADD LAT/LON GRID LINES 
# (set thickness of grid lines) 
ThinLineThickness    = 0.2
MediumLineThickness  = 0.3
ThickLineThickness   = 0.6
# (and what multiples of lat/lon have lines)
ThinLineFrequency    = 0.1
MediumLineFrequency  = 0.5
ThickLineFrequency   = 1.0
# alright, lets add those lines
AddGridlines(ax, ThinLineThickness, MediumLineThickness, ThickLineThickness,
                 ThinLineFrequency,  MediumLineFrequency,  ThickLineFrequency, StandOutColour)

# set the plot limits to the same as the BACKGROUND TOPO MAP
plt.xlim([LonMin, LonMax])
plt.ylim([LatMin, LatMax])



In [ ]:
# ---------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------
XVarName      = 'meanlon'    # string name of the x-axis variable
YVarName      = 'meanlat'      # string name of the y-axis variable
DensityColourMap = plt.cm.plasma          # colourmap for the density plot
XBins         = 100                       # number of bins in x
YBins         = 100                       # number of bins in y
PlotColourStyle = 'Dark'                  # 'Light' or 'Dark'
# ---------------------------------------------------------------

# Pull the two variables and flatten to 1D
XData = FeatureXR[XVarName].values.flatten()
YData = FeatureXR[YVarName].values.flatten()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict([~np.isnan(XData),~np.isnan(YData)])